In [ ]:
#========================================================================
# Name: calc_wrf_gamma_properties.ipynb
# Author: McKenna W. Stanford
# Author Contact: mckenna.stanford@pnnl.gov
#
# Description: Extract WRF gamma distribution parameters for cloud droplets
#              and precipitation for 3km CACTI domain.
#========================================================================

In [2]:
# Imports
#===============================
import xarray as xr
import numpy as np
import glob
import datetime
import time
import os
import dask
from dask.distributed import wait
from distributed import Client, LocalCluster
#from functions import find_nearest, to_datetime
import pickle
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.spatial import cKDTree
import scipy
from scipy.special import gamma
from scipy.special import gammaincinv
from scipy.special import gammaincc
import dask.array as da
import numba

import warnings
warnings.filterwarnings("ignore")

In [3]:
read_in = True

In [4]:
if read_in:
    read_path = '/pscratch/sd/m/mckenna/'
    file_name = read_path+'wrf_matched_file_lists.p'
    with open(file_name, 'rb') as f:
        wrf_files_dict = pickle.load(f)
    wrf_ref_files = wrf_files_dict['wrf_ref_files']
    wrf_2d_files = wrf_files_dict['wrf_2d_files']
    wrf_3d_files = wrf_files_dict['wrf_3d_files']
    wrf_ref_datetimes = wrf_files_dict['ref_file_datetimes']
    wrf_2d_datetimes = wrf_files_dict['wrf_2d_datetimes']
    wrf_3d_datetimes = wrf_files_dict['wrf_3d_datetimes']

    num_wrf_2d_files = len(wrf_2d_files)
    num_wrf_3d_files = len(wrf_3d_files)
    num_wrf_ref_files = len(wrf_ref_files)
    print('# of 2D files:',num_wrf_2d_files)
    print('# of 3D files:',num_wrf_3d_files)
    print('# of ref files:',num_wrf_ref_files)

# of 2D files: 8794
# of 3D files: 8794
# of ref files: 8794


In [5]:
def subset_var_3d(data,mask):

    nz = len(data[:,0,0])
    # mask values where the csapr mask == 0.
    reshaped_data_3d = []
    
    for kk in range(nz):
        data_2d = data[kk,:,:]
        masked_data = np.ma.masked_array(data_2d, mask=mask == 0)
        #print(aaaaa)
        
        # Extract the 2D subset defined by the mask
        # The unmasked elements will define the dimensions
        unmasked_data = masked_data[~masked_data.mask]  # Extract non-masked data
        
        
        row_indices, col_indices = np.where(~masked_data.mask)  # Get the unmasked indices
        
        new_rows = len(np.unique(row_indices))  # Number of unique rows in the unmasked region
        new_cols = len(np.unique(col_indices))  # Number of unique columns in the unmasked region
        
        reshaped_data = unmasked_data.reshape((new_rows, new_cols))
    
        reshaped_data_3d.append(reshaped_data.data)

    reshaped_data_3d = np.asarray(reshaped_data_3d)
    
    return reshaped_data_3d

In [6]:
def subset_var_2d(data,mask):

    # mask values where the csapr mask == 0.
    masked_data = np.ma.masked_array(data, mask=mask == 0)

    # Extract the 2D subset defined by the mask
    # The unmasked elements will define the dimensions
    unmasked_data = masked_data[~masked_data.mask]  # Extract non-masked data


    row_indices, col_indices = np.where(~masked_data.mask)  # Get the unmasked indices

    new_rows = len(np.unique(row_indices))  # Number of unique rows in the unmasked region
    new_cols = len(np.unique(col_indices))  # Number of unique columns in the unmasked region

    reshaped_data = unmasked_data.reshape((new_rows, new_cols))

    return reshaped_data.data

In [7]:
def get_csapr_mask(wrf_ref_file, wrf_2d_file):
    # Open datasets
    ds_ref = xr.open_dataset(wrf_ref_file)
    lon_ref = ds_ref['XLONG'].squeeze().values
    lat_ref = ds_ref['XLAT'].squeeze().values
    ds_ref.close()

    ds_2d = xr.open_dataset(wrf_2d_file)
    lon_2d = ds_2d['CAC_LONG'].squeeze().values
    lat_2d = ds_2d['CAC_LAT'].squeeze().values
    ds_2d.close()

    # Flatten the 2D lat/lon arrays into a list of (lon, lat) points
    ref_points = np.column_stack((lon_ref.ravel(), lat_ref.ravel()))
    wrf_points = np.column_stack((lon_2d.ravel(), lat_2d.ravel()))

    # Build a KDTree from the reference WRF grid (fast spatial lookup)
    tree = cKDTree(ref_points)

    # Query the WRF 2D grid against the reference grid
    _, indices = tree.query(wrf_points, distance_upper_bound=1e-6)  # Use a small tolerance

    # Initialize mask with zeros
    csapr_mask = np.zeros_like(lon_2d, dtype=float)

    # Only mark valid matches (ignore out-of-bounds results)
    valid_matches = indices < ref_points.shape[0]
    csapr_mask.ravel()[valid_matches] = 1.0

    return csapr_mask,lon_2d,lat_2d


In [8]:
def get_terrain_height(csapr_mask):
    ter_path = '/global/cfs/projectdirs/m1657/avarble/cacti/wrf/output/2019-05-01/'
    ter_file = ter_path+'wrfout_d01_2019-05-01_00:00:00'
    ds_ter = xr.open_dataset(ter_file,engine='netcdf4')
    ter_hgt = ds_ter['HGT'].squeeze()
    ds_ter.close()
    ter_hgt = subset_var_2d(ter_hgt,csapr_mask)

    return ter_hgt

# Get some static variables that are only needed once

In [9]:
#=======================# Get height from different file#=======================ds_ref = xr.open_dataset(wrf_ref_files[0])zamsl = ds_ref['ZAMSL'].values.squeeze()ref = ds_ref['REFL_10CM'].values.squeeze()ds_ref.close()nz,ny,nx = zamsl.shapez_agl = np.zeros((nz,ny,nx))csapr_mask,lon_2d,lat_2d = get_csapr_mask(wrf_ref_files[0],wrf_2d_files[0])ter_hgt = get_terrain_height(csapr_mask)for kk in range(nz):    z_agl[kk,:,:] = zamsl[kk,:,:] - ter_hgtnp.savez("generated_data/static_vars.npz",z_agl=z_agl,zamsl=zamsl,ter_hgt=ter_hgt,lon_2d=lon_2d,lat_2d=lat_2d,csapr_mask=csapr_mask)


In [10]:
np.shape(zamsl)

(80, 77, 77)

In [14]:
zamsl[:,44,44]

array([  791.5162,   892.253 ,   995.7251,  1101.7986,  1210.442 ,
        1321.6824,  1435.5887,  1552.2201,  1671.6471,  1793.9413,
        1919.1937,  2047.4934,  2178.9417,  2313.6428,  2451.6885,
        2593.1548,  2738.1816,  2886.9473,  3039.5295,  3196.0815,
        3356.9607,  3522.2278,  3692.16  ,  3866.9988,  4046.8877,
        4231.955 ,  4422.0825,  4617.295 ,  4817.8857,  5023.998 ,
        5235.833 ,  5453.8135,  5678.5195,  5910.3867,  6149.4863,
        6395.981 ,  6649.8994,  6911.123 ,  7179.6978,  7455.9355,
        7740.2363,  8032.9536,  8334.451 ,  8645.231 ,  8965.937 ,
        9297.235 ,  9639.8   ,  9994.313 , 10361.49  , 10741.822 ,
       11135.954 , 11544.588 , 11967.677 , 12403.45  , 12851.573 ,
       13312.1455, 13785.814 , 14272.983 , 14773.823 , 15289.668 ,
       15810.24  , 16328.4   , 16845.633 , 17363.895 , 17882.008 ,
       18396.93  , 18915.965 , 19439.59  , 19967.188 , 20497.506 ,
       21029.234 , 21562.682 , 22096.691 , 22631.2   , 23165.9

In [15]:
z_agl[:,44,44]

array([    0.        ,   100.73681641,   204.20892334,   310.28240967,
         418.92584229,   530.16619873,   644.0725708 ,   760.70391846,
         880.13092041,  1002.42510986,  1127.67749023,  1255.97729492,
        1387.42553711,  1522.12670898,  1660.17236328,  1801.63867188,
        1946.66552734,  2095.43115234,  2248.01342773,  2404.56542969,
        2565.44458008,  2730.71166992,  2900.64379883,  3075.48266602,
        3255.37158203,  3440.43896484,  3630.56640625,  3825.77880859,
        4026.36962891,  4232.48193359,  4444.31689453,  4662.29736328,
        4887.00341797,  5118.87060547,  5357.97021484,  5604.46484375,
        5858.38330078,  6119.60693359,  6388.18164062,  6664.41943359,
        6948.72021484,  7241.4375    ,  7542.93505859,  7853.71533203,
        8174.42041016,  8505.71875   ,  8848.28320312,  9202.796875  ,
        9569.97363281,  9950.30566406, 10344.4375    , 10753.07128906,
       11176.16015625, 11611.93359375, 12060.05664062, 12520.62890625,
      

In [10]:
def get_3d_vars(wrf_3d_file,z_agl,zamsl,csapr_mask,lon_2d,lat_2d):

    #=======================
    # Get 3D variables
    #=======================
    ds_3d = xr.open_dataset(wrf_3d_file)
    temp = ds_3d['CAC_T'].values.squeeze()-273.15 # deg C
    pressure = ds_3d['CAC_P'].values.squeeze() # Pa
    qc = ds_3d['CAC_QC'].values.squeeze()*1.e-3 # kg/kg
    qi = ds_3d['CAC_QI'].values.squeeze()*1.e-3 # kg/kg
    qs = ds_3d['CAC_QS'].values.squeeze()*1.e-3 # kg/kg
    qr = ds_3d['CAC_QR'].values.squeeze()*1.e-3 # kg/kg 
    qg = ds_3d['CAC_QG'].values.squeeze()*1.e-3 # kg/kg
    qv = ds_3d['CAC_QV'].values.squeeze()*1.e-3 # kg/kg
    nc = ds_3d['CAC_NC'].values.squeeze() # /kg
    ni = ds_3d['CAC_NI'].values.squeeze() # /kg
    nr = ds_3d['CAC_NR'].values.squeeze() # /kg
    w = ds_3d['CAC_W'].values.squeeze() # /kg
    t_str = ds_3d['Times'].values[0].decode('utf-8') # Time byte string
    ds_3d.close()
    
    # Convert to datetime
    time_dt = datetime.datetime.strptime(t_str, "%Y-%m-%d_%H:%M:%S").replace(tzinfo=datetime.timezone.utc)
    
    #=======================
    # Trim to CSAPR domain
    #=======================    
    temp = subset_var_3d(temp,csapr_mask)
    qc = subset_var_3d(qc,csapr_mask)
    qr = subset_var_3d(qr,csapr_mask)
    qi = subset_var_3d(qi,csapr_mask)
    qs = subset_var_3d(qs,csapr_mask)
    qg = subset_var_3d(qg,csapr_mask)
    nc = subset_var_3d(nc,csapr_mask)
    ni = subset_var_3d(ni,csapr_mask)
    nr = subset_var_3d(nr,csapr_mask)
    qv = subset_var_3d(qv,csapr_mask)
    w = subset_var_3d(w,csapr_mask)
    pressure = subset_var_3d(pressure,csapr_mask)
    lon_2d = subset_var_2d(lon_2d,csapr_mask)
    lat_2d = subset_var_2d(lat_2d,csapr_mask)

    Rd=287.04
    Tv = (temp+237.15)*(1. + 0.61*qv)
    rho_air = pressure/(Rd*Tv)
    

    x3d_dict = {'qc':qc,\
                'qr':qr,\
                'qi':qi,\
                'qs':qs,\
                'qg':qg,\
                'nc':nc,\
                'nr':nr,\
                'ni':ni,\
                'rho_air':rho_air,\
                'z_agl':z_agl,\
                'zamsl':zamsl,\
                'temp':temp,\
                'pressure':pressure,\
                'ter_hgt':ter_hgt,\
                'w':w,\
                'lon':lon_2d,\
                'lat':lat_2d,\
                'time_dt':time_dt,\
               }

    return x3d_dict

In [11]:
def compute_gamma_dsd_parameters_liquid_3d(q, N, rho_air, species, mu=0.0, rho_w=1000.0, q_min_threshold=1e-12, N_min_threshold=1e-6):

    from scipy.special import gamma
    from scipy.special import gammaincinv
    from scipy.special import gammaincc
    """
    Computes DSD diagnostic parameters for a 3D array of liquid species (rain, cloud, etc.) 
    including median mass diameter, reflectivity, and both intercept and slope parameters.
    
    Parameters:
    - q: 3D array of liquid species water mixing ratio [kg/kg]
    - N: 3D array of liquid species number concentration [#/kg]
    - rho_air: 3D array of moist air density [kg/m^3]
    - mu: shape parameter of the gamma distribution (default = 0 for exponential)
    - rho_w: density of water [kg/m^3] (default = 1000)
    - q_min_threshold: Minimum threshold for q to perform the calculation (default = 1e-12 kg/m^3, per Thompson scheme)
    - N_min_threshold: Minimum threshold for N to perform the calculation (default = 1e-6 /m^3, per Thompson scheme)
    
    Returns:
    - Dictionary containing 3D arrays for N_0 (intercept), lambda (slope), D_num, D_vol, D_mass, 
      spectral widths, skewnesses, median mass diameter, and reflectivity (Z)
    """

    # Apply threshold condition directly to q_liquid and N_liquid arrays
    valid_mask = (q*rho_air > q_min_threshold) & (N*rho_air > N_min_threshold)

    # Initialize arrays with NaN values
    N_0 = np.full_like(q, np.nan, dtype=np.float64)
    log_N_0 = np.full_like(q, np.nan, dtype=np.float64)
    lambda_ = np.full_like(q, np.nan, dtype=np.float64)
    D_num = np.full_like(q, np.nan, dtype=np.float64)
    D_vol = np.full_like(q, np.nan, dtype=np.float64)
    D_mass = np.full_like(q, np.nan, dtype=np.float64)
    #sigma_num = np.full_like(q, np.nan, dtype=np.float64)
    #skew_num = np.full_like(q, np.nan, dtype=np.float64)
    #kurt_num = np.full_like(q, np.nan, dtype=np.float64)
    #sigma_mass = np.full_like(q, np.nan, dtype=np.float64)
    #skew_mass = np.full_like(q, np.nan, dtype=np.float64)
    #kurt_mass = np.full_like(q, np.nan, dtype=np.float64)
    median_mass_diameter = np.full_like(q, np.nan, dtype=np.float64)
    Z = np.full_like(q, np.nan, dtype=np.float64)
    dBZ = np.full_like(q, np.nan, dtype=np.float64)
    q_gt_100um = np.full_like(q,np.nan,dtype=np.float64)
    N_gt_100um = np.full_like(q,np.nan,dtype=np.float64)


    # Save original variables
    q_orig = q.copy()
    N_orig = N.copy()
    rho_air_orig = rho_air.copy()

    
    # For the valid indices (where q and N are above threshold)
    q = q[valid_mask]
    N = N[valid_mask]
    rho_air = rho_air[valid_mask]

    # If cloud, need to compute mu; it will be a 3D array; if rain, mu is an integer
    if species == 'cloud':
        dum = 1000.e6 / (N*rho_air) + 2
        dum_int = np.round(dum).astype(int)
        mu = np.minimum(15, dum_int)
        

    # Compute lambda and N_0 based on q_liquid and N_liquid
    am = (np.pi * rho_w) / 6.0 # pre-factor in m-D relationship for liquid [kg]
    bm = 3. # exponent in m-D relatinship for liquid

    easy_option = False
    if easy_option:
        lambda_ij = (gamma(mu + bm + 1) / gamma(mu + 1) * (N / q) * am) ** (1 / bm) # [m^-1]
        N_0_ij = (N*rho_air)*lambda_ij ** (mu + 1) / (gamma(mu + 1)) # [m^-4]
    else:
        if species == 'cloud':
            Nt_c_max = 1999.E6
            D0c = 1.e-6
            D0r = 50.e-6
            rc = q*rho_air
            tmp_N = np.maximum(2.,np.minimum(N*rho_air,Nt_c_max))
            lamc = (tmp_N*gamma(mu + bm + 1)*am/rc/gamma(mu + 1))**(1./bm)
            xDc = (bm + mu + 1.)/lamc
            dumid1 = np.where(xDc < D0c)
            dumid2 = np.where(xDc > D0r*2.)
            if np.size(dumid1) > 0.:
                dumid1 = dumid1[0]
                lamc[dumid1] = (bm + mu[dumid1] + 1.)/D0c
            elif np.size(dumid2) > 0.:
                dumid2 = dumid2[0]
                lamc[dumid2] = (bm + mu[dumid2] + 1.)/(D0r*2.)
            tmp_N = np.minimum(Nt_c_max,gamma(mu + 1)/gamma(mu + bm + 1)*rc/am*lamc**bm)
            
            lambda_ij = lamc
            N_0_ij = tmp_N*lambda_ij**(mu+1)/gamma(mu+1.)
        
        else:
            # Rain stays easy
            lambda_ij = (gamma(mu + bm + 1) / gamma(mu + 1) * (N / q) * am) ** (1 / bm) # [m^-1]
            N_0_ij = (N*rho_air)*lambda_ij ** (mu + 1) / (gamma(mu + 1)) # [m^-4]
        
    log_N_0_ij = np.log(N_0_ij)


    # Compute moments M0 to M7
    M = {}
    for n in range(8):
        M[n] = N_0_ij * gamma(mu + n + 1) / lambda_ij ** (mu + n + 1)

    # Number-weighted statistics (D_num, D_vol, sigma_num, skew_num, kurt_num)
    D_num_ij = M[1] / M[0]
    D_vol_ij = (M[3] / M[0]) ** (1/3)
    
    # Central moments (number-weighted)
    mu2_num = M[2] / M[0] - D_num_ij**2
    mu3_num = M[3] / M[0] - 3 * D_num_ij * mu2_num - D_num_ij**3
    mu4_num = M[4] / M[0] - 4 * D_num_ij * mu3_num - 6 * D_num_ij**2 * mu2_num - D_num_ij**4
    
    #sigma_num_ij = np.sqrt(mu2_num)
    #skew_num_ij = mu3_num / sigma_num_ij**3
    #kurt_num_ij = mu4_num / sigma_num_ij**4 - 3.0  # excess kurtosis
    
    # Mass-weighted statistics (D_mass, sigma_mass, skew_mass, kurt_mass)
    D_mass_ij = M[4] / M[3]
    
    # Central moments (mass-weighted)
    mu2_mass = M[5] / M[3] - D_mass_ij**2
    mu3_mass = M[6] / M[3] - 3 * D_mass_ij * mu2_mass - D_mass_ij**3
    mu4_mass = M[7] / M[3] - 4 * D_mass_ij * mu3_mass - 6 * D_mass_ij**2 * mu2_mass - D_mass_ij**4
    
    #sigma_mass_ij = np.sqrt(mu2_mass)
    #skew_mass_ij = mu3_mass / sigma_mass_ij**3
    #kurt_mass_ij = mu4_mass / sigma_mass_ij**4 - 3.0  # excess kurtosis

    # Median mass diameter using incomplete gamma function
    median_mass_diameter_ij = gammaincinv(mu + bm + 1, 0.5) / lambda_ij # m

    # Compute the 6th moment for reflectivity (Z)
    M_6 = N_0_ij * gamma(mu + 6 + 1) / lambda_ij ** (mu + 6 + 1) # m^6/m^3
    K = 1.e18
    Z_ij = K*M_6  # mm^6/m^3
    dBZ_ij = 10.*np.log10(Z_ij)

    # Mass & number concentration above 100 um
    D_thresh = 100.*1.e-6 # 100 um in meters
    s_num = mu + 1.
    s_mass = mu + bm + 1.
    x = lambda_ij * D_thresh
    
    # Number concentration above 100 microns [#/m^3]
    N_gt_100um_ij = N_0_ij / lambda_ij**s_num * gamma(s_num) * gammaincc(s_num, x)
    
    # Mass concentration above 100 microns [kg/m^3]
    q_gt_100um_ij = N_0_ij * am / lambda_ij**s_mass * gamma(s_mass) * gammaincc(s_mass, x)

    # Store values in the valid array positions
    N_0[valid_mask] = N_0_ij
    log_N_0[valid_mask] = log_N_0_ij
    lambda_[valid_mask] = lambda_ij
    D_num[valid_mask] = D_num_ij
    D_vol[valid_mask] = D_vol_ij
    D_mass[valid_mask] = D_mass_ij
    #sigma_num[valid_mask] = sigma_num_ij
    #skew_num[valid_mask] = skew_num_ij
    #kurt_num[valid_mask] = kurt_num_ij
    #sigma_mass[valid_mask] = sigma_mass_ij
    #skew_mass[valid_mask] = skew_mass_ij
    #kurt_mass[valid_mask] = kurt_mass_ij
    median_mass_diameter[valid_mask] = median_mass_diameter_ij
    Z[valid_mask] = Z_ij
    dBZ[valid_mask] = dBZ_ij
    q_gt_100um[valid_mask] = q_gt_100um_ij/rho_air
    N_gt_100um[valid_mask] = N_gt_100um_ij/rho_air
    
    #q_orig = np.full_like(N_0, np.nan, dtype=np.float64)
    #N_orig = np.full_like(N_0, np.nan, dtype=np.float64)
    #rho_air_orig = np.full_like(N_0, np.nan, dtype=np.float64)
    #q_orig[valid_mask] = q
    #N_orig[valid_mask] = N
    #rho_air_orig[valid_mask] = rho_air

    M_full = {}
    for key in M:
        M_full[key] = np.full_like(N_0,np.nan,dtype=np.float64)
        M_full[key][valid_mask] = M[key]
    
    if species == 'cloud':
        tmp_mu = np.full_like(N_0,np.nan)
        tmp_mu[valid_mask] = mu
        mu = tmp_mu

    return {
        'N_0': N_0,  # intercept parameter [m^-4]
        'log_N_0': log_N_0,  # Log of intercept parameter [log(m^-4)]
        'lambda': lambda_,  # Slope parameter [m^-1]
        'D_num': D_num,  # Number-weighted mean diameter [m]
        'D_vol': D_vol,  # Volume-weighted mean diameter [m]
        'D_mass': D_mass,  # Mass-weighted mean diameter [m]
        #'sigma_num': sigma_num,  # Number-weighted standard deviation [m]
        #'skew_num': skew_num,  # Number-weighted skewness [-]
        #'kurt_num': kurt_num,  # Number-weighted kurtosis [-]
        #'sigma_mass': sigma_mass,  # Mass-weighted standard deviation [-]
        #'skew_mass': skew_mass,  # Mass-weighted skewness [-]
        #'kurt_mass': kurt_mass,  # Mass-weighted kurtosis [-]
        'mmd': median_mass_diameter,  # Median mass diameter [m]
        'Z': Z,  # Linear Reflectivity [mm^6 m^-3]
        'dBZ': dBZ,  # Reflectivity [dBZ]
        'mu':mu, # Shape parameter (3D for cloud, integer for rain) [-]
        'q':q_orig, # prongostic mass mixing ratio [kg/kg]
        'N':N_orig, # prognsotic total number concentraiton mixing ratio [#/kg]
        'rho_air':rho_air_orig, #  moist air density [kg/m^3]
        'q_gt_100um':q_gt_100um, # mass mixing ratio for diameters > 100 um [kg/kg] 
        'N_gt_100um':N_gt_100um, # number concentration mixing ratio for diameters > 100 um [#/kg] 
        'M_dict':M_full, # dictionary of raw moments
    }

In [12]:
def write_file(x3d_dict,rain_dict,cloud_dict,liq_dict):
    out_dict = {'D_num':liq_dict['D_num'],
            'D_vol':liq_dict['D_vol'],
            'D_mass':liq_dict['D_mass'],
            #'sigma_num':liq_dict['sigma_num'],
            #'skew_num':liq_dict['skew_num'],
            #'kurt_num':liq_dict['kurt_num'],
            #'sigma_mass':liq_dict['sigma_mass'],
            #'skew_mass':liq_dict['skew_mass'],
            #'kurt_mass':liq_dict['kurt_mass'],
            'mmd':liq_dict['mmd'],
            'N_gt_100um':liq_dict['N_gt_100um'],
            'q_gt_100um':liq_dict['q_gt_100um'],
            'q':liq_dict['q'],
            'N':liq_dict['N'],
            'rho_air':liq_dict['rho_air'],
            'N_0_r':rain_dict['N_0'],
            'N_0_c':cloud_dict['N_0'],
            'lambda_r':rain_dict['lambda'],
            'lambda_c':cloud_dict['lambda'],
            'mu_c':cloud_dict['mu'],
            'D_num_c':cloud_dict['D_num'],
            'D_vol_c':cloud_dict['D_vol'],
            'D_mass_c':cloud_dict['D_mass'],
            'mmd_c':cloud_dict['mmd'],
            'D_num_r':rain_dict['D_num'],
            'D_vol_r':rain_dict['D_vol'],
            'D_mass_r':rain_dict['D_mass'],
            'mmd_r':rain_dict['mmd'],
            #'sigma_num_c':cloud_dict['sigma_num'],
            #'skew_num_c':cloud_dict['skew_num'],
            #'kurt_num_c':cloud_dict['kurt_num'],
            #'sigma_mass_c':cloud_dict['sigma_mass'],
            #'skew_mass_c':cloud_dict['skew_mass'],
            #'kurt_mass_c':cloud_dict['kurt_mass'],
            #'sigma_num_r':rain_dict['sigma_num'],
            #'skew_num_r':rain_dict['skew_num'],
            #'kurt_num_r':rain_dict['kurt_num'],
            #'sigma_mass_r':rain_dict['sigma_mass'],
            #'skew_mass_r':rain_dict['skew_mass'],
            #'kurt_mass_r':rain_dict['kurt_mass'],
            'q_r_gt_100um':rain_dict['q_gt_100um'],
            'N_r_gt_100um':rain_dict['N_gt_100um'],
            'q_c_gt_100um':cloud_dict['q_gt_100um'],
            'N_c_gt_100um':cloud_dict['N_gt_100um'],
               }

    # Define variable attributes
    out_dict_attrs = {
                'D_num': {
                    'long_name': 'Number-weighted Mean Diameter',
                    'units': 'm',
                },
                'D_vol':{
                    'long_name': 'Mean Volumetric Diameter',
                    'units':'m',
                },
                'D_mass':{
                    'long_name': 'Mass-weighted Mean Diameter',
                    'units':'m',
                },
                #'sigma_num':{
                #    'long_name': 'Number-weighted DSD Standard Deviation',
                #    'units':'m',
                #    'description_1':'To derive relative dispersion, divide sigma_num by D_num',
                #},
                #'skew_num':{
                #    'long_name': 'Number-weighted DSD Skewness',
                #    'units':'unitless',
                #},
                #'kurt_num':{
                #    'long_name': 'Number-weighted DSD Kurtosis',
                #    'units':'Dimensionless',
                #},
                #'sigma_mass':{
                #    'long_name': 'Mass-weighted DSD Standard Deviation',
                #    'units':'m',
                #    'description_1':'To derive relative dispersion, divide sigma_num by D_mass',
                #},
                #'skew_mass':{
                #    'long_name': 'Mass-weighted DSD Skewness',
                #    'units':'unitless',
                #},
                #'kurt_mass':{
                #    'long_name': 'Mass-weighted DSD Kurtosis',
                #    'units':'Dimensionless',
                #},
                'mmd':{
                    'long_name': 'DSD Median Mass Diameter',
                    'units':'m',
                },
                'N_gt_100um':{
                    'long_name': 'Total Number Concentration (Cloud + Rain) Mixing Ratio for Diameters > 100 um',
                    'units':'kg^-1',
                },
                'q_gt_100um':{
                    'long_name': 'Total Mass (Cloud + Rain) Mixing Ratio for Diameters > 100 um',
                    'units':'kg kg^-1',
                },
                'N':{
                    'long_name': 'Total Number Concentration (Cloud + Rain) Mixing Ratio',
                    'units':'kg^-1',
                },
                'q':{
                    'long_name': 'Total Mass (Cloud + Rain) Mixing Ratio',
                    'units':'kg kg^-1',
                },
                'rho_air':{
                    'long_name': 'Moist Air Density',
                    'units':'kg m^-3',
                },
                'N_0_r':{
                    'long_name': 'Rain Intercept Parameter',
                    'units':'m^-4',
                },
                'N_0_c':{
                    'long_name': 'Cloud Intercept Parameter',
                    'units':'m^-4',
                },
                'lambda_r':{
                    'long_name': 'Rain Slope Parameter',
                    'units':'m^-1',
                },
                'lambda_c':{
                    'long_name': 'Cloud Slope Parameter',
                    'units':'m^-1',
                },
                'mu_c':{
                    'long_name': 'Cloud Shape Parameter',
                    'units':'unitless',
                },
                'D_num_c':{
                    'long_name': 'Cloud Number-weighted Mean Diameter',
                    'units':'m',
                },
                'D_vol_c':{
                    'long_name': 'Cloud Volumetric Mean Diameter',
                    'units':'m',
                },
                'D_mass_c':{
                    'long_name': 'Cloud Mass-weighted Mean Diameter',
                    'units':'m',
                },
                'mmd_c':{
                    'long_name': 'Cloud Median Mass Diameter',
                    'units':'m',
                },
                'D_num_r':{
                    'long_name': 'Rain Number-weighted Mean Diameter',
                    'units':'m',
                },
                'D_vol_r':{
                    'long_name': 'Rain Volumetric Mean Diameter',
                    'units':'m',
                },
                'D_mass_r':{
                    'long_name': 'Rain Mass-weighted Mean Diameter',
                    'units':'m',
                },
                'mmd_r':{
                    'long_name': 'Rain Median Mass Diameter',
                    'units':'m',
                },
                #'sigma_num_c':{
                #    'long_name': 'Cloud Number-weighted Standard Deviation',
                #    'units':'m',
                #    'description_1':'To derive relative dispersion, divide sigma_num_c by D_num_c',
                #},
                #'skew_num_c':{
                #    'long_name': 'Cloud Number-weighted Skewness',
                #    'units':'unitless',
                #},
                #'kurt_num_c':{
                #    'long_name': 'Cloud Number-weighted Kurtosis',
                #    'units':'unitless',
                #},
                #'sigma_mass_c':{
                #    'long_name': 'Cloud Mass-weighted Standard Deviation',
                #    'units':'m',
                #    'description_1':'To derive relative dispersion, divide sigma_mass_c by D_mass_c',
                #},
                #'skew_mass_c':{
                #    'long_name': 'Cloud Mass-weighted Skewness',
                #    'units':'unitless',
                #},
                #'kurt_mass_c':{
                #    'long_name': 'Cloud Mass-weighted Kurtosis',
                #    'units':'unitless',
                #},
                #'sigma_num_r':{
                #    'long_name': 'Rain Number-weighted Standard Deviation',
                #    'units':'m',
                #    'description_1':'To derive relative dispersion, divide sigma_num_r by D_num_r',
                #},
                #'skew_num_r':{
                #    'long_name': 'Rain Number-weighted Skewness',
                #    'units':'unitless',
                #},
                #'kurt_num_r':{
                #    'long_name': 'Rain Number-weighted Kurtosis',
                #    'units':'unitless',
                #},
                #'sigma_mass_r':{
                #    'long_name': 'Rain Mass-weighted Standard Deviation',
                #    'units':'m',
                #    'description_1':'To derive relative dispersion, divide sigma_mass_r by D_mass_r',
                #},
                #'skew_mass_r':{
                #    'long_name': 'Rain Mass-weighted Skewness',
                #    'units':'unitless',
                #},
                #'kurt_mass_r':{
                #    'long_name': 'Rain Mass-weighted Kurtosis',
                #    'units':'unitless',
                #},
                'q_r_gt_100um':{
                    'long_name': 'Rain Mass Mixing Ratio for Diameters > 100 um',
                    'units':'kg/kg',
                },
                'N_r_gt_100um':{
                    'long_name': 'Rain Number Concentration Mixing Ratio for Diameters > 100 um',
                    'units':'/kg',
                },
                'q_c_gt_100um':{
                    'long_name': 'Cloud Mass Mixing Ratio for Diameters > 100 um',
                    'units':'kg/kg',
                },
                'N_c_gt_100um':{
                    'long_name': 'Cloud Number Concentration Mixing Ratio for Diameters > 100 um',
                    'units':'/kg',
                },

    }   

    date_str = x3d_dict['time_dt'].strftime('%Y%m%d_%H:%M:%S')
    # Output parameters
    output_path = '/pscratch/sd/m/mckenna/cacti/wrf/derived_3km_dsd_parameters/'
    output_filename = f'{output_path}WRF_CACTI_3km_DSD_parameters_liq_only_'+date_str+'.nc'

    # Define dimensions
    time_dimname = 'time'
    lon_dimname = 'west_east'
    lat_dimname = 'south_north'
    z_dimname = 'bottom_top'

    var_dict = {}
    # Define output variable dictionary
    for key, value in out_dict.items():
        if np.ndim(value) == 3.:
            var_dict[key] = ([z_dimname,lat_dimname,lon_dimname],value,out_dict_attrs[key])
            
    wrf_epoch = np.asarray([int(x3d_dict['time_dt'].timestamp())])

    # Define coordinate attributes
    coord_attr_dict = {'time':{'long_name':'WRF Epoch time','units':'Epoch'},
                       'lon':{'long_name':'Longitude','units':'degrees'},
                       'lat':{'long_name':'Latitude','units':'degrees'},
                       'zamsl':{'long_name':'Height Above Mean Sea Level','units':'m'},
                      }
    # Define coordinates
    coord_dict = {
            'time': ([time_dimname],wrf_epoch,coord_attr_dict['time']),
            'lon': ([lat_dimname,lon_dimname],x3d_dict['lon'],coord_attr_dict['lon']),
            'lat': ([lat_dimname,lon_dimname],x3d_dict['lat'],coord_attr_dict['lat']),
            'zamsl': ([z_dimname,lat_dimname,lon_dimname],x3d_dict['zamsl'],coord_attr_dict['zamsl']),
            }

    # Define global attributes
    gattr_dict = {
        'title':  'WRF derived 3-km liquid DSD parameters for CACTI campaign (10-15-2018 - 03/03/2019)', \
        'Institution': 'Pacific Northwest National Laboratoy', \
        'Contact': 'McKenna Stanford, mckenna.stnaford@pnnl.gov', \
        'Created_on':  time.ctime(time.time()), \
        'Date': date_str, \
    }

    # Define xarray dataset
    dsout = xr.Dataset(var_dict, coords=coord_dict, attrs=gattr_dict)
    
    # Delete file if it already exists
    if os.path.isfile(output_filename):
        os.remove(output_filename)
    
    # Set encoding/compression for all variables
    comp = dict(zlib=True)
    encoding = {var: comp for var in dsout.data_vars}
    
    # Write to netcdf file
    dsout.to_netcdf(path=output_filename, mode="w",
                    format="NETCDF4", unlimited_dims=time_dimname, encoding=encoding)

    return output_filename

In [13]:
def compute_liquid_dsd_parameters(cloud_dict, rain_dict, rho_air, q_min_threshold=1e-12, N_min_threshold=1e-6):
    """
    Computes combined liquid (cloud water + rain) DSD metrics, including DSD moments, number- and -mass-weighted mean diamters, 
    median mass diameter, and reflectivity.

    
    Parameters:
    - cloud_dict: dictionary holding all derived DSD variables for cloud water species
    - rain_dict: dictionary holding all derived DSD variables for rain species
    - rho_air: 3D array of moist air density [kg/m^3]
    
    Returns:
    - Dictionary containing 3D arrays for D_num, D_vol, D_mass, 
      spectral widths, skewnesses, kurtosis, median mass diameter, and reflectivity (Z)
    """

    # Combine rain and cloud species data
    M_comb = {n: np.nansum([rain_dict['M_dict'][n], cloud_dict['M_dict'][n]], axis=0) for n in rain_dict['M_dict']}
    q_comb = np.nansum([rain_dict['q'],cloud_dict['q']],axis=0)
    N_comb = np.nansum([rain_dict['N'],cloud_dict['N']],axis=0)
    Z_comb = np.nansum([rain_dict['Z'],cloud_dict['Z']],axis=0)
    #N_r = rain_dict['N']*rho_air
    #N_c = cloud_dict['N']*rho_air
    #q_r = rain_dict['q']*rho_air
    #q_c = cloud_dict['q']*rho_air
    #print('np.nanmax(N_comb) [m^-3]:',np.nanmax(N_comb*rho_air))
    #print('np.nanmax(N_r) [m^-3]:',np.nanmax(N_r))
    #print('np.nanmax(N_c) [m^-3]:',np.nanmax(N_c))
    #print('np.nanmax(q_comb) [g m^-3]:',np.nanmax(q_comb*rho_air*1.e3))
    #print('np.nanmax(q_r) [g m^-3]:',np.nanmax(q_r)*1.e3)
    #print('np.nanmax(q_c) [g m^-3]:',np.nanmax(q_c)*1.e3)
    #print('')
    
    # Extract parameters
    lambda_c, N_0_c, mu_c = cloud_dict['lambda'], cloud_dict['N_0'], cloud_dict['mu']
    lambda_r, N_0_r, mu_r = rain_dict['lambda'], rain_dict['N_0'], rain_dict['mu']

    # Constants
    rho_w = 1000. # kg/m^3
    a_r = a_c = np.pi*rho_w/6.
    b_r = b_c = 3.

    rho_air_orig = rho_air.copy()
    
    #valid_mask = ~np.isnan(M_comb[0])  # True where M0 is valid (not NaN)
    #valid_mask = np.where(q_comb > 1.e-4)
    #valid_mask = np.where(q_comb*rho_air > 1.e-12)
    valid_mask = np.where(q_comb > 1.e-12)

    # Apply mask    
    N_0_r, N_0_c = N_0_r[valid_mask], N_0_c[valid_mask]
    lambda_r, lambda_c = lambda_r[valid_mask], lambda_c[valid_mask]
    mu_c = mu_c[valid_mask]
    rho_air = rho_air[valid_mask]
    Z_ij = Z_comb[valid_mask]
    #N_r = N_r[valid_mask]
    #q_r = q_r[valid_mask]
    #N_c = N_c[valid_mask]
    #q_c = q_c[valid_mask]
    #print('np.nanmax(N_r) [m^-3]:',np.nanmax(N_r))
    #print('np.nanmax(N_c) [m^-3]:',np.nanmax(N_c))
    #print('np.nanmax(q_r) [g/m^-3]:',np.nanmax(q_r)*1.e3)
    #print('np.nanmax(q_c) [g/m^3]:',np.nanmax(q_c)*1.e3)
    #print('')

    # Initialize arrays with NaN values
    D_num = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    D_vol = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    D_mass = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    #sigma_num = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    #skew_num = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    #kurt_num = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    #sigma_mass = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    #skew_mass = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    #kurt_mass = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    median_mass_diameter = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    Z = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    dBZ = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    N_gt_100um = np.full_like(M_comb[0], np.nan, dtype=np.float64)
    q_gt_100um = np.full_like(M_comb[0], np.nan, dtype=np.float64)

    # Extract valid combined moments
    M_comb_ij = {}
    for key in M_comb:
        M_comb_ij[key] = M_comb[key][valid_mask]

    # Number-weighted statistics (D_num, D_vol, sigma_num, skew_num, kurt_num)
    D_num_ij = M_comb_ij[1] / M_comb_ij[0]
    D_vol_ij = (M_comb_ij[3] / M_comb_ij[0]) ** (1/3)
    
    # Central moments (number-weighted)
    mu2_num = M_comb_ij[2] / M_comb_ij[0] - D_num_ij**2
    mu3_num = M_comb_ij[3] / M_comb_ij[0] - 3 * D_num_ij * mu2_num - D_num_ij**3
    mu4_num = M_comb_ij[4] / M_comb_ij[0] - 4 * D_num_ij * mu3_num - 6 * D_num_ij**2 * mu2_num - D_num_ij**4
    
    #sigma_num_ij = np.sqrt(mu2_num)
    #skew_num_ij = mu3_num / sigma_num_ij**3
    #kurt_num_ij = mu4_num / sigma_num_ij**4 - 3.0  # excess kurtosis
    
    # Mass-weighted statistics (D_mass, sigma_mass, skew_mass, kurt_mass)
    D_mass_ij = M_comb_ij[4] / M_comb_ij[3]
    
    # Central moments (mass-weighted)
    mu2_mass = M_comb_ij[5] / M_comb_ij[3] - D_mass_ij**2
    mu3_mass = M_comb_ij[6] / M_comb_ij[3] - 3 * D_mass_ij * mu2_mass - D_mass_ij**3
    mu4_mass = M_comb_ij[7] / M_comb_ij[3] - 4 * D_mass_ij * mu3_mass - 6 * D_mass_ij**2 * mu2_mass - D_mass_ij**4
    
    #sigma_mass_ij = np.sqrt(mu2_mass)
    #skew_mass_ij = mu3_mass / sigma_mass_ij**3
    #kurt_mass_ij = mu4_mass / sigma_mass_ij**4 - 3.0  # excess kurtosis
    
    # Median mass diameter
    D_bins = np.arange(-0.5,10001.5,1)*1.e-6 # [m]
    D = np.convolve(D_bins, np.ones(2) / 2, 'valid')  # Midpoint of D bins
    dD = np.diff(D_bins)[0]  # constant bin width

    # Broadcast to 2D arrays: (num_grid_points, num_D)
    D_broadcast = D[None, :]  # Shape: (1, num_D)
    
    # Set chunk size (adjust as needed to avoid memory overload)
    chunk_size = 10000
    num_grid_points = N_0_r.shape[0]
    print('# of grid points:',num_grid_points)
    
    # Allocate arrays for chunk calculations
    median_mass_diameter_ij, qtot = np.full(num_grid_points, np.nan), np.full(num_grid_points, np.nan)   
    N_gt_100um_ij,q_gt_100um_ij = np.full(num_grid_points, np.nan), np.full(num_grid_points, np.nan)


    # All of the following calculated outside of loop for efficiency
    D_power_mu_b_r = D_broadcast**(b_r + mu_r)
    D_power_mu_r = D_broadcast**(mu_r)
    D_power_b_c = D_broadcast**b_c
    N_0_r = np.nan_to_num(N_0_r, nan=0.0)
    N_0_c = np.nan_to_num(N_0_c, nan=0.0)
    lambda_r = np.nan_to_num(lambda_r, nan=1e12)
    lambda_c = np.nan_to_num(lambda_c, nan=1e12)
    mu_c = np.nan_to_num(mu_c, nan=0.0)

    # Threshold Size
    threshold_size = 100.*1.e-6 # 100 um, in meters
    threshold_mask = D_broadcast > threshold_size   


    for i_start in range(0, num_grid_points, chunk_size):
        #print('------------------------')
        i_end = min(i_start + chunk_size, num_grid_points)

        # Progress print
        print(f"Progress: {100 * i_end / num_grid_points:.1f}%", end='\r')
    
        # Slice chunks
        N_0_r_chunk = N_0_r[i_start:i_end, None]
        N_0_c_chunk = N_0_c[i_start:i_end, None]
        lambda_r_chunk = lambda_r[i_start:i_end, None]
        lambda_c_chunk = lambda_c[i_start:i_end, None]
        mu_c_chunk = mu_c[i_start:i_end, None]

        D_power_mu_b_c = D_power_b_c * D_broadcast**mu_c_chunk
        D_power_mu_c = D_broadcast**mu_c_chunk
        # Compute M_r and M_c for this chunk
        M_r_chunk = a_r * N_0_r_chunk * D_power_mu_b_r * np.exp(-lambda_r_chunk * D_broadcast)
        M_c_chunk = a_c * N_0_c_chunk * D_power_mu_b_c * np.exp(-lambda_c_chunk * D_broadcast)
        # Compute N_r and N_c for this chunk
        N_r_chunk = N_0_r_chunk * D_power_mu_r * np.exp(-lambda_r_chunk * D_broadcast)
        N_c_chunk = N_0_c_chunk * D_power_mu_c * np.exp(-lambda_c_chunk * D_broadcast)
        #N_r_chunk_2 = N_r[i_start:i_end]
        #N_c_chunk_2 = N_c[i_start:i_end]
        #M_r_chunk_2 = q_r[i_start:i_end]
        #M_c_chunk_2 = q_c[i_start:i_end]

        #int_N_r_chunk = np.trapz(N_r_chunk,D,axis=1)
        #int_N_c_chunk = np.trapz(N_c_chunk,D,axis=1)
        #int_M_r_chunk = np.trapz(M_r_chunk,D,axis=1)
        #int_M_c_chunk = np.trapz(M_c_chunk,D,axis=1)
 
        #print('np.nanmax(N_r_chunk_2) [m^-3]:',np.nanmax(N_r_chunk_2))
        #print('np.nanmax(int_N_r_chunk) [m^-3]:',np.nanmax(int_N_r_chunk))
        #print('np.nanmax(N_c_chunk_2) [m^-3]:',np.nanmax(N_c_chunk_2))
        #print('np.nanmax(int_N_c_chunk) [m^-3]:',np.nanmax(int_N_c_chunk))
        #print('')
        #print('np.nanmax(M_r_chunk_2) [g m^-3]:',np.nanmax(M_r_chunk_2)*1.e3)
        #print('np.nanmax(int_M_r_chunk) [g m^-3]:',np.nanmax(int_M_r_chunk)*1.e3)
        #print('np.nanmax(M_c_chunk_2) [g m^-3]:',np.nanmax(M_c_chunk_2)*1.e3)
        #print('np.nanmax(int_M_c_chunk) [g m^-3]:',np.nanmax(int_M_c_chunk)*1.e3)
        #print('')
        
        M_total_chunk = M_r_chunk + M_c_chunk
    
        # Example: Compute cumulative mass distribution and median mass diameter
        M_cumsum = np.cumsum(M_total_chunk * dD, axis=1)
        M_total_integrated = M_cumsum[:, -1]
        qtot[i_start:i_end] = M_total_integrated
        mask = M_total_integrated > 0
        M_fraction = M_cumsum[mask] / M_total_integrated[mask, None]

        # Find median diameter index
        D_median_chunk = np.full(i_end - i_start, np.nan)
        idx_median = np.argmax(M_fraction >= 0.5,axis=1)
        D_median_chunk[mask] = D[idx_median]
        median_mass_diameter_ij[i_start:i_end] = D_median_chunk



        # Calculate number concentration for particles > 100 um
        N_r_gt_100um_chunk = np.sum(N_r_chunk * threshold_mask * dD, axis=1)
        N_c_gt_100um_chunk = np.sum(N_c_chunk * threshold_mask * dD, axis=1)
        N_gt_100um_ij[i_start:i_end] = N_r_gt_100um_chunk + N_c_gt_100um_chunk
        #print('np.nanmax(N_gt_100um_ij):',np.nanmax(N_gt_100um_ij))

        # Calculate mass for particles > 100 µm
        q_r_gt_100um_chunk = np.sum(M_r_chunk * threshold_mask * dD, axis=1)
        q_c_gt_100um_chunk = np.sum(M_c_chunk * threshold_mask * dD, axis=1)
        q_gt_100um_ij[i_start:i_end] = q_r_gt_100um_chunk + q_c_gt_100um_chunk


    
    # Store values in the valid array positions
    D_num[valid_mask] = D_num_ij
    D_vol[valid_mask] = D_vol_ij
    D_mass[valid_mask] = D_mass_ij
    #sigma_num[valid_mask] = sigma_num_ij
    #skew_num[valid_mask] = skew_num_ij
    #kurt_num[valid_mask] = kurt_num_ij
    #sigma_mass[valid_mask] = sigma_mass_ij
    #skew_mass[valid_mask] = skew_mass_ij
    #kurt_mass[valid_mask] = kurt_mass_ij
    median_mass_diameter[valid_mask] = median_mass_diameter_ij
    N_gt_100um[valid_mask] = N_gt_100um_ij/rho_air
    #print('np.nanmax(N_gt_100um_ij/rho_air):',np.nanmax(N_gt_100um_ij/rho_air))
    q_gt_100um[valid_mask] = q_gt_100um_ij/rho_air
    Z[valid_mask] = Z_ij
    dBZ[valid_mask] = 10.*np.log10(Z_ij)

    
    return {
        'D_num': D_num,  # Number-weighted mean diameter [m]
        'D_vol': D_vol,  # Volume-weighted mean diameter [m]
        'D_mass': D_mass,  # Mass-weighted mean diameter [m]
        #'sigma_num': sigma_num,  # Number-weighted standard deviation [m]
        #'skew_num': skew_num,  # Number-weighted skewness [-]
        #'kurt_num': kurt_num,  # Number-weighted kurtosis [-]
        #'sigma_mass': sigma_mass,  # Mass-weighted standard deviation [-]
        #'skew_mass': skew_mass,  # Mass-weighted skewness [-]
        #'kurt_mass': kurt_mass,  # Mass-weighted kurtosis [-]
        'mmd': median_mass_diameter,  # Median mass diameter [m]
        'N_gt_100um': N_gt_100um,  # Number concentration mixing ratio for diameters > 100 um [kg^-1]
        'q_gt_100um': q_gt_100um,  # Mass mixing ratio for diameters > 100 um [kg kg^-1]
        'Z': Z,  # Linear Reflectivity [mm^6 m^-3]
        'dBZ': dBZ,  # Reflectivity [dBZ]
        'q':q_comb, # Sum of cloud and rain mass mixing ratios [kg/kg]
        'N':N_comb, # Sum of cloud and rain number concentraiton mixing ratio [#/kg]
        'rho_air':rho_air_orig, # moist air density [kg/m^3]
        'M_dict':M_comb, # dictionary of raw moments
    }

# Test in serial

In [14]:
%%time
if True:
    dumi = 643
    x3d_dict = get_3d_vars(wrf_3d_files[dumi],z_agl,zamsl,csapr_mask,lon_2d,lat_2d)
    rain_dict = compute_gamma_dsd_parameters_liquid_3d(x3d_dict['qr'], x3d_dict['nr'], x3d_dict['rho_air'], species='rain', mu=0.0, rho_w=1000.0, q_min_threshold=1e-12, N_min_threshold=1e-6)
    cloud_dict = compute_gamma_dsd_parameters_liquid_3d(x3d_dict['qc'], x3d_dict['nc'], x3d_dict['rho_air'], species='cloud', mu=0.0, rho_w=1000.0, q_min_threshold=1e-12, N_min_threshold=1e-6)
    liq_dict = compute_liquid_dsd_parameters(cloud_dict, rain_dict, x3d_dict['rho_air'], q_min_threshold=1e-12, N_min_threshold=1e-6)
    outfile_name = write_file(x3d_dict,rain_dict,cloud_dict,liq_dict)
    #outfile_name = driver_func(wrf_3d_files[dumi])

# of grid points: 16436
CPU times: user 14 s, sys: 2.36 s, total: 16.3 s
Wall time: 16.5 s


# Dask stuff

In [15]:
dask.config.config["distributed"]["dashboard"]["link"] = "{JUPYTERHUB_SERVICE_PREFIX}proxy/{host}:{port}/status" 

In [16]:
iparallel = True
if iparallel:
    cluster = LocalCluster(n_workers=20,threads_per_worker=1,memory_limit='20GB')#,dashboard_address=':8787')
    client = Client(cluster)

In [17]:
cluster

LocalCluster(eaa2c73f, 'tcp://127.0.0.1:46643', workers=20, threads=20, memory=372.53 GiB)

In [18]:
def suppress_warnings():
    import warnings
    warnings.filterwarnings("ignore", message=".*pyproj unable to set PROJ database path.*", category=UserWarning)
    warnings.filterwarnings("ignore", category=RuntimeWarning)

client.run(suppress_warnings)

{'tcp://127.0.0.1:32903': None,
 'tcp://127.0.0.1:32919': None,
 'tcp://127.0.0.1:33113': None,
 'tcp://127.0.0.1:33381': None,
 'tcp://127.0.0.1:33755': None,
 'tcp://127.0.0.1:33945': None,
 'tcp://127.0.0.1:34683': None,
 'tcp://127.0.0.1:35221': None,
 'tcp://127.0.0.1:36189': None,
 'tcp://127.0.0.1:38093': None,
 'tcp://127.0.0.1:38251': None,
 'tcp://127.0.0.1:39321': None,
 'tcp://127.0.0.1:39871': None,
 'tcp://127.0.0.1:41661': None,
 'tcp://127.0.0.1:42573': None,
 'tcp://127.0.0.1:43161': None,
 'tcp://127.0.0.1:43945': None,
 'tcp://127.0.0.1:44027': None,
 'tcp://127.0.0.1:44773': None,
 'tcp://127.0.0.1:45959': None}

In [19]:
#cluster.scale(n=14)

In [20]:
def driver_func(wrf_3d_file):    # Load in static variables    static = np.load("generated_data/static_vars.npz")    z_agl = static["z_agl"]    zamsl = static["zamsl"]    csapr_mask = static["csapr_mask"]    lon_2d = static["lon_2d"]    lat_2d = static["lat_2d"]        x3d_dict = get_3d_vars(wrf_3d_file,z_agl,zamsl,csapr_mask,lon_2d,lat_2d)    rain_dict = compute_gamma_dsd_parameters_liquid_3d(x3d_dict['qr'], x3d_dict['nr'], x3d_dict['rho_air'], species='rain', mu=0.0, rho_w=1000.0, q_min_threshold=1e-12, N_min_threshold=1e-6)    cloud_dict = compute_gamma_dsd_parameters_liquid_3d(x3d_dict['qc'], x3d_dict['nc'], x3d_dict['rho_air'], species='cloud', mu=0.0, rho_w=1000.0, q_min_threshold=1e-12, N_min_threshold=1e-6)    liq_dict = compute_liquid_dsd_parameters(cloud_dict, rain_dict, cloud_dict['rho_air'], q_min_threshold=1e-12, N_min_threshold=1e-6)    outfile_name = write_file(x3d_dict,rain_dict,cloud_dict,liq_dict)    #outfile_name = ''    #dum = x3d_dict    #dum = liq_dict    return outfile_name


In [21]:
%%time
results = []
start_id = 0
#end_id = 100
end_id = len(wrf_3d_files)
for ii in range(start_id,end_id):
    result = dask.delayed(driver_func)(wrf_3d_files[ii])
    results.append(result)

CPU times: user 366 ms, sys: 108 ms, total: 474 ms
Wall time: 380 ms


In [22]:
%%time
futures = client.compute(results)  # results is a list of delayed or dask collections

CPU times: user 394 ms, sys: 16.8 ms, total: 411 ms
Wall time: 406 ms


In [23]:
%%time
# Trigger dask computation
#with warnings.catch_warnings():
#    warnings.simplefilter("ignore", category=RuntimeWarning)
#final_result = dask.compute(*results)
final_result = client.gather(futures)
print("Computation done.")
#wait(final_result)

/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-pac

# of grid points: 131741
# of grid points: 5765
# of grid points: 220
# of grid points: 2127
# of grid points: 40633
# of grid points: 1306
# of grid points: 9287
# of grid points: 2313
# of grid points: 42709
# of grid points: 0
# of grid points: 7
# of grid points: 0
# of grid points: 2
# of grid points: 151245
# of grid points: 0
# of grid points: 551
# of grid points: 1741
# of grid points: 0
# of grid points: 1
# of grid points: 55981
# of grid points: 28565
# of grid points: 88124
# of grid points: 9230
# of grid points: 0
# of grid points: 0
# of grid points: 5448
# of grid points: 103
# of grid points: 0
# of grid points: 0
# of grid points: 97
# of grid points: 158028
# of grid points: 0
# of grid points: 0
# of grid points: 16103
# of grid points: 16453
# of grid points: 12
# of grid points: 40140
# of grid points: 39783
# of grid points: 2252
# of grid points: 23
# of grid points: 7903
# of grid points: 0
# of grid points: 6931
# of grid points: 1249
# of grid points: 0
# of

In [23]:
len(final_result)

5000

In [24]:
final_result[0]

'/pscratch/sd/m/mckenna/cacti/wrf/derived_3km_dsd_parameters/WRF_CACTI_3km_DSD_parameters_liq_only_20181015_00:00:00.nc'

In [37]:
# Gracefully close previous client and cluster if they exist
try:
    client.close()
except NameError:
    pass

try:
    cluster.close()
except NameError:
    pass

# of grid points: 36534
# of grid points: 1030
# of grid points: 1869
# of grid points: 151931
# of grid points: 15728
# of grid points: 56486
# of grid points: 92557
 0ogress: 100.0%
# of grid points: 966
# of grid points: 26483
# of grid points: 92467
 4916ess: 100.0%
# of grid points: 0
# of grid points: 1076
# of grid points: 0
# of grid points: 106669
# of grid points: 104070
# of grid points: 13074
# of grid points: 3864
# of grid points: 154194
# of grid points: 55000
# of grid points: 0
# of grid points: 6721
# of grid points: 47114
# of grid points: 37940
# of grid points: 10
# of grid points: 30698
# of grid points: 71079
# of grid points: 1350
# of grid points: 58
# of grid points: 5038
# of grid points: 100974
# of grid points: 82849
# of grid points: 0
# of grid points: 0
# of grid points: 17
# of grid points: 2925
# of grid points: 49
# of grid points: 71847
# of grid points: 0
# of grid points: 43459
# of grid points: 0
# of grid points: 0
# of grid points: 97607
# of gr